In [1]:
import os

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from dataset import PairGaitDataset
from model import SiameseGaitVerifier


# Hardcoded configuration for Jupyter/local execution
TRAIN_PATH = "../data/split_casia_b/train/"
VAL_PATH = "../data/split_casia_b/val/"
CHECKPOINT_PATH = "best_gait_verifier.pth"

SEQUENCE_LENGTH = 64
BATCH_SIZE = 16
EPOCHS = 25
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2

POSITIVES_PER_ANCHOR = 2
NEGATIVES_PER_ANCHOR = 2
EMBEDDING_DIM = 128
HIDDEN_DIM = 256

THRESHOLD = 0.5
PATIENCE = 5


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total = 0
    correct = 0

    with torch.no_grad():
        for x1, x2, y in loader:
            x1 = x1.to(device)
            x2 = x2.to(device)
            y = y.to(device).float().unsqueeze(1)

            logits, _, _ = model(x1, x2)
            loss = criterion(logits, y)

            probs = torch.sigmoid(logits)
            preds = (probs >= THRESHOLD).float()

            total_loss += loss.item() * x1.size(0)
            total += x1.size(0)
            correct += (preds == y).sum().item()

    return total_loss / max(1, total), correct / max(1, total)


def main():
    if not os.path.exists(TRAIN_PATH):
        raise FileNotFoundError(f"TRAIN_PATH not found: {TRAIN_PATH}")
    if not os.path.exists(VAL_PATH):
        raise FileNotFoundError(f"VAL_PATH not found: {VAL_PATH}")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on: {device}")

    train_dataset = PairGaitDataset(
        data_path=TRAIN_PATH,
        sequence_length=SEQUENCE_LENGTH,
        positives_per_anchor=POSITIVES_PER_ANCHOR,
        negatives_per_anchor=NEGATIVES_PER_ANCHOR,
        is_training=True,
    )
    val_dataset = PairGaitDataset(
        data_path=VAL_PATH,
        sequence_length=SEQUENCE_LENGTH,
        positives_per_anchor=POSITIVES_PER_ANCHOR,
        negatives_per_anchor=NEGATIVES_PER_ANCHOR,
        is_training=False,
    )

    if len(train_dataset) == 0 or len(val_dataset) == 0:
        raise RuntimeError("Empty train/val pair dataset. Check folder structure and .pkl files.")

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    model = SiameseGaitVerifier(
        num_nodes=17,
        in_channels=8,
        embedding_dim=EMBEDDING_DIM,
        hidden_dim=HIDDEN_DIM,
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    best_val_loss = float("inf")
    trigger_times = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss = 0.0
        train_total = 0
        train_correct = 0

        for x1, x2, y in train_loader:
            x1 = x1.to(device)
            x2 = x2.to(device)
            y = y.to(device).float().unsqueeze(1)

            optimizer.zero_grad()
            logits, _, _ = model(x1, x2)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            probs = torch.sigmoid(logits)
            preds = (probs >= THRESHOLD).float()

            train_loss += loss.item() * x1.size(0)
            train_total += x1.size(0)
            train_correct += (preds == y).sum().item()

        avg_train_loss = train_loss / max(1, train_total)
        train_acc = train_correct / max(1, train_total)

        avg_val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        print(
            f"Epoch [{epoch}/{EPOCHS}] | "
            f"Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.4f} | "
            f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.4f}"
        )

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            trigger_times = 0
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_val_loss": best_val_loss,
                },
                CHECKPOINT_PATH,
            )
            print(f"New best model saved: {CHECKPOINT_PATH} (val_loss={best_val_loss:.4f})")
        else:
            trigger_times += 1
            print(f"No improvement. Patience: {trigger_times}/{PATIENCE}")
            if trigger_times >= PATIENCE:
                print("Early stopping triggered.")
                break
if __name__ == "__main__":
    main()


Training on: cuda
PairGaitDataset: identities=99, pairs=43552, training=True
PairGaitDataset: identities=13, pairs=5716, training=False
Epoch [1/25] | Train Loss: 0.6727, Train Acc: 0.5704 | Val Loss: 0.6539, Val Acc: 0.6520
New best model saved: best_gait_verifier.pth (val_loss=0.6539)
Epoch [2/25] | Train Loss: 0.5874, Train Acc: 0.6899 | Val Loss: 0.5710, Val Acc: 0.7302
New best model saved: best_gait_verifier.pth (val_loss=0.5710)
Epoch [3/25] | Train Loss: 0.5607, Train Acc: 0.7109 | Val Loss: 0.5892, Val Acc: 0.7232
No improvement. Patience: 1/5
Epoch [4/25] | Train Loss: 0.5455, Train Acc: 0.7234 | Val Loss: 0.5780, Val Acc: 0.7350
No improvement. Patience: 2/5
Epoch [5/25] | Train Loss: 0.5249, Train Acc: 0.7390 | Val Loss: 0.5564, Val Acc: 0.7444
New best model saved: best_gait_verifier.pth (val_loss=0.5564)
Epoch [6/25] | Train Loss: 0.4999, Train Acc: 0.7528 | Val Loss: 0.5643, Val Acc: 0.7481
No improvement. Patience: 1/5
Epoch [7/25] | Train Loss: 0.4834, Train Acc: 0.764